# 表現力を高めたGaussian Process

このNotebookでは `JointEncoderGP` (DKL)、`InfiniteWidthBNNGP`、`SpectralMixtureGP`、`SingleTaskDeepGP` の位置づけと基本APIを比較します。モデルごとに表現力の源と学習方法が異なる点を確認します。

In [ ]:
import torch
from robotorchan.models import InfiniteWidthBNNGP, JointEncoderGP, SpectralMixtureGP
from robotorchan.models.deep_gp import SingleTaskDeepGP

torch.manual_seed(0)
train_X = torch.rand(12, 2, dtype=torch.double)
train_Y = torch.sin(2 * torch.pi * train_X[:, :1]) + 0.2 * train_X[:, 1:2]


## Exact GPとして扱える表現力モデル

DKLは入力表現をニューラルネットワークで学習します。Infinite-width BNN GPはニューラルネットワーク由来の解析kernelを使い、Spectral Mixture GPは周波数構造をkernelで表現します。3モデルともExact GPのposterior contractを維持します。

In [ ]:
models = {
    "DKL": JointEncoderGP(train_X, train_Y, latent_dim=2, hidden_dims=(6,), random_state=0),
    "I-BNN": InfiniteWidthBNNGP(train_X, train_Y, depth=2),
    "SM": SpectralMixtureGP(train_X, train_Y, num_mixtures=2),
}

test_X = torch.linspace(0, 1, 8, dtype=torch.double)
test_X = torch.stack([test_X, torch.full_like(test_X, 0.5)], dim=-1)
for name, model in models.items():
    model.eval()
    model.likelihood.eval()
    posterior = model.posterior(test_X)
    print(name, posterior.mean.shape, posterior.variance.shape)


## Deep Gaussian Process

DeepGPは中間表現も確率変数です。そのためExact GPとは学習objectiveとposterior samplingが異なります。ここでは構築とposterior contractを確認し、重いtraining loopはbenchmarkや専用テストに任せます。

In [ ]:
deep_gp = SingleTaskDeepGP(
    train_X,
    train_Y,
    hidden_dims=(3,),
    num_inducing=5,
    posterior_samples=16,
    random_state=0,
)
deep_gp.eval()
posterior = deep_gp.posterior(test_X)
print("DeepGP", posterior.mean.shape, posterior.variance.shape)


## 使い分け

- **DKL**: 目的に有用な非線形特徴をデータから学習したい場合
- **DeepGP**: 階層的で確率的な潜在表現を必要とする場合
- **Infinite-width BNN GP**: neural-network由来priorとExact GP推論を両立したい場合
- **Spectral Mixture GP**: 周期・準周期・複数周波数を持つ定常構造を表現したい場合

表現力が高いモデルを自動的に選ぶのではなく、標準 `SingleTaskGP` と予測精度、不確実性校正、計算時間、BO性能を比較して選択します。